In [ ]:
import gc

try:
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception:
    gc.collect()
    print('Torch not available, skip GPU cache clear.')


# 第 4 课：Reranking / 重排序

前面我们已经学了：
- RAG 基本主线
- Chunking
- Embedding 与向量检索

这一课进入真实 RAG 里非常常见的一步：

**Reranking，也就是重排序。**

它解决的问题是：

**第一轮检索找回来的候选不一定排得最合理，所以还要再精排一次。**


## 1. 为什么向量检索之后还不够

向量检索通常擅长的是：
- 快速召回一批“可能相关”的 chunk

但它不一定总能把最相关的 chunk 排在最前面。

原因很多，比如：
- embedding 更偏语义接近，而不一定抓住问句里的关键约束
- chunk 里虽然主题相关，但不一定真正回答了问题
- 多个 chunk 都有点相关，但强相关程度不同

所以真实系统里很常见的流程是：

1. 第一轮召回：先取 `top-k`
2. 第二轮精排：再从这批候选里挑出真正最该喂给模型的前几个


## 2. 什么是 Reranker

`Reranker` 的作用就是：

**对 query 和候选 chunk 做更细致的匹配评分。**

你可以把它理解成：

- 检索器像海选
- reranker 像复试

第一轮先把可能相关的人都叫进来，
第二轮再认真比较谁最适合。


## 3. 这节课怎么讲 reranking

真实项目里，reranker 常常会用：
- cross-encoder
- late interaction 模型
- 更强的 query-document scoring 模型

但这一课我们还是先用教学版思路，把主线讲透：

- 第一轮：简单 embedding / 相似度召回
- 第二轮：更细粒度的 query-chunk 匹配打分

重点是理解为什么“召回”和“精排”要分成两步。


In [ ]:
import math
import random
import re
from collections import Counter

random.seed(42)


In [ ]:
chunks = [
    'RAG retrieves external knowledge before generation.',
    'Chunking splits documents into smaller passages for retrieval.',
    'Embeddings map text into vectors so semantic similarity can be measured.',
    'Evaluation checks retrieval quality, groundedness, and final answer correctness.',
    'Vector databases store embeddings and support similarity search at scale.',
    'Reranking reorders retrieved candidates so the best evidence appears at the top.',
    'A reranker often compares the query and each candidate passage more carefully than the first-stage retriever.'
]

for i, chunk in enumerate(chunks, start=1):
    print(f'chunk {i}: {chunk}')


## 4. 先做第一轮召回

我们沿用上一课的教学版 embedding：
- 每个 token 一个小向量
- 文本向量 = token 向量平均
- 用余弦相似度做检索

这样先得到第一轮 `top-k` 候选。


In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z]+", text.lower())


token_to_vec = {}
embedding_dim = 8

def get_token_vector(token):
    if token not in token_to_vec:
        rnd = random.Random(hash(token) & 0xffffffff)
        token_to_vec[token] = [rnd.uniform(-1, 1) for _ in range(embedding_dim)]
    return token_to_vec[token]


def embed_text(text):
    tokens = tokenize(text)
    if not tokens:
        return [0.0] * embedding_dim
    vecs = [get_token_vector(token) for token in tokens]
    return [sum(v[i] for v in vecs) / len(vecs) for i in range(embedding_dim)]


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


chunk_embeddings = [embed_text(chunk) for chunk in chunks]

def retrieve_with_embeddings(query, chunks, chunk_embeddings, top_k=5):
    query_embedding = embed_text(query)
    scored = []
    for chunk, emb in zip(chunks, chunk_embeddings):
        score = cosine_similarity(query_embedding, emb)
        scored.append((score, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


query = 'Why do we need a reranker after retrieval?'
retrieved = retrieve_with_embeddings(query, chunks, chunk_embeddings, top_k=5)

print('第一轮召回结果:')
for score, chunk in retrieved:
    print(f'score={score:.3f}')
    print(chunk)
    print()


## 5. Reranking 在做什么

第一轮召回后，我们通常只剩下一小批候选，比如：
- top-20
- top-50
- top-100

这个时候就可以用一个更贵、但更准的方法去重新比较它们。

也就是说：
- 第一轮追求“快”
- 第二轮追求“准”


## 6. 一个教学版 reranker

为了讲清主线，这里我们做一个简化 reranker：

- 它会更关注 query 和 chunk 之间的词重叠
- 同时给某些关键术语更多权重

真实 cross-encoder 会更强，因为它会把 query 和 chunk 拼在一起做联合编码。
但第一课先不用把模型做复杂。


In [ ]:
important_terms = {
    'rerank': 3.0,
    'retrieve': 2.0,
    'query': 2.0,
    'candidate': 2.0,
    'evidence': 2.0,
    'top': 1.5,
    'best': 1.5
}

def normalize_token(token):
    token = token.lower()
    if token.startswith('rerank'):
        return 'rerank'
    if token.startswith('retriev'):
        return 'retrieve'
    if token.startswith('candidate'):
        return 'candidate'
    return token

def simple_reranker_score(query, chunk):
    q_tokens = [normalize_token(t) for t in tokenize(query)]
    c_tokens = [normalize_token(t) for t in tokenize(chunk)]

    c_counter = Counter(c_tokens)
    score = 0.0
    for token in q_tokens:
        if token in c_counter:
            score += important_terms.get(token, 1.0) * c_counter[token]

    # 给 query 和 chunk 都同时出现的关键短语一点额外奖励
    q_text = query.lower()
    c_text = chunk.lower()
    if ('reranker' in q_text or 'reranking' in q_text) and ('reranker' in c_text or 'reranking' in c_text):
        score += 2.0
    if ('retrieval' in q_text or 'retrieve' in q_text) and ('retrieval' in c_text or 'retrieved' in c_text):
        score += 1.5
    if 'best evidence' in c_text:
        score += 2.5

    return score


reranked = []
for first_stage_score, chunk in retrieved:
    rerank_score = simple_reranker_score(query, chunk)
    reranked.append((rerank_score, first_stage_score, chunk))

reranked.sort(key=lambda x: x[0], reverse=True)

print('第二轮重排序结果:')
for rerank_score, first_stage_score, chunk in reranked:
    print(f'rerank_score={rerank_score:.3f} | first_stage_score={first_stage_score:.3f}')
    print(chunk)
    print()


## 7. 现在你可以观察什么

你应该会看到一个很重要的现象：

- 第一轮召回找到的是“可能相关”的候选
- 第二轮 reranking 更强调“谁最直接回答 query”

这就是 RAG 里“召回”和“精排”分工不同的原因。


## 8. 真实项目里的 reranker 通常长什么样

真实项目里常见的 reranker 通常会比这节课复杂很多，比如：

- `cross-encoder`
  直接把 query 和 chunk 一起送进模型评分

- `LLM-as-reranker`
  让大模型判断哪个候选更相关

- `hybrid reranking`
  结合 embedding 分数、BM25 分数、业务规则等

但你要记住，核心思想没有变：

**先粗召回，再精排序。**


In [ ]:
top_2_for_generation = reranked[:2]

print('最终送给生成模型的 top-2:')
for rerank_score, first_stage_score, chunk in top_2_for_generation:
    print(f'rerank_score={rerank_score:.3f}')
    print(chunk)
    print()


## 9. Evaluation 视角下怎么判断 reranking 有没有帮助

判断 reranking 是否有价值，通常会看：

- 正确 chunk 的排名有没有更靠前
- top-1 / top-3 命中率有没有更高
- 最终回答质量有没有提升
- 模型是不是更 grounded，幻觉有没有减少

也就是说，reranking 不是“看起来高级”，而是要真的让正确证据更早进入 prompt。


In [ ]:
gold_chunk = 'Reranking reorders retrieved candidates so the best evidence appears at the top.'

first_stage_rank = next(i for i, (_, chunk) in enumerate(retrieved, start=1) if chunk == gold_chunk)
second_stage_rank = next(i for i, (_, _, chunk) in enumerate(reranked, start=1) if chunk == gold_chunk)

print('gold chunk 在第一轮召回中的排名:', first_stage_rank)
print('gold chunk 在第二轮 reranking 中的排名:', second_stage_rank)


## 10. 这节课最该记住什么

如果你想抓住这节课的核心主线，就记住：

- 第一轮检索负责“尽量别漏掉”
- 第二轮 reranking 负责“把最相关的排到前面”

所以 reranking 的本质是：

**在一批候选里做更认真、更昂贵、但也更准确的相关性判断。**


## 11. 下一课最自然学什么

学完这一课后，最自然的下一步就是：

**Hybrid Search / 混合检索**

因为真实 RAG 里常常不会只靠 embedding，一般还会把稀疏检索和稠密检索结合起来。
